In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abdulsalamramatu/100-feature-response-sample/total_response_sample.xlsx
/kaggle/input/datasets/abdulsalamramatu/complete-math-features/new_math_df.csv


In [2]:
!pip install replicate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, cohen_kappa_score
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

import time
from kaggle_secrets import UserSecretsClient
#import replicate
from typing import Dict, List


# STEP 1: 100 response per feature extraction

In [3]:
math_df=pd.read_csv("/kaggle/input/datasets/abdulsalamramatu/complete-math-features/new_math_df.csv")
mtld_na=math_df.loc[math_df.mtld.isna()].conversation_id.tolist()
math_df=math_df[~math_df['conversation_id'].isin(mtld_na)]

math_df["mtld_log"] = np.log1p(math_df["mtld"])
math_df["word_count_log"] = np.log1p(math_df["word_count"])

In [5]:
def sample_score_bins(df, score, n_total=100, n_bins=4):
    bin_edges=np.percentile(df[score], np.linspace(0,100, n_bins+1))
    bin_edges[0]=-np.inf
    bin_edges[-1]=np.inf
    df['score_bin']=pd.cut(
        df[score], 
        bins=bin_edges, 
        labels=[f'Bin{i+1}' for i  in range(n_bins)],
        include_lowest=True)
    
    samples_per_bin=n_total// n_bins
    reminder=n_total % n_bins
    
    sampled_dfs=[]
    
    for i, bin_label in enumerate(df['score_bin'].cat.categories):
        
        bin_df=df[df['score_bin']==bin_label]
        n_samples=samples_per_bin +(1 if i < reminder else 0)
        n_samples=min(n_samples, len(bin_df))
        
        if n_samples>0:
            sampled_dfs.append(bin_df.sample(n=n_samples, random_state=42))

    sampled_df = pd.concat(sampled_dfs)
    cols_to_keep=['tutor', 'conversation_id', 'student_mistake',  'tutors_response']

    columns_to_keep = list(set(cols_to_keep ))
    sampled_df = sampled_df[columns_to_keep]
    sampled_df['conversation_id'] = sampled_df['conversation_id'].astype(str)

    #sampled_df = sampled_df.drop(columns=['score_bin'])
    
    return sampled_df, bin_edges

In [6]:
features = ['PressReasoning_prob', 'PressAccuracy_prob', 'Uptake_prob', 'politeness_score', 'agency_score']
sample_dict={}
bin_edges_dict={}
for feature in features:
    print(f"Sampling for {feature}....")
    sampled_df, edges= sample_score_bins(math_df, feature,n_total=100, n_bins=4)
    sample_dict[feature]=sampled_df
    bin_edges_dict[feature]=edges   
    sampled_df.to_csv(f"sampled_{feature}.csv", index=False)

Sampling for PressReasoning_prob....
Sampling for PressAccuracy_prob....
Sampling for Uptake_prob....
Sampling for politeness_score....
Sampling for agency_score....


In [7]:
math_df

,tutor,conversation_id,student_mistake,tutors_response,PressReasoning_prob,PressAccuracy_prob,Uptake_prob,politeness_score,agency_score,flesch_reading_ease,...,word_count,flesch_score,flesch_ease,Mistake_Identification,Mistake_Location,Actionability,Providing_Guidance,mtld_log,word_count_log,score_bin
0,Sonnet,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,"Great, you've correctly identified the cost of...",0.000816,0.009659,0.558105,0.975984,0.577917,52.050000,...,26,52.0500,16.0,2,2,2,2,3.393613,3.295837,Bin4
1,Llama318B,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,Now that we know the cost of 1 pound of meat i...,0.000762,0.005924,0.908691,0.951331,0.540315,80.097647,...,32,80.0976,6.0,2,1,1,1,3.606584,3.496508,Bin3
2,Llama31405B,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,"You're close, but I notice that you calculated...",0.004723,0.757324,0.013527,0.907285,0.487226,47.376429,...,39,47.3764,18.0,2,2,2,2,3.483349,3.688879,Bin1
3,GPT4,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,"That's correct. So, if 1 pound of meat costs $...",0.001601,0.690430,0.047394,0.986556,0.510410,98.252500,...,27,98.2525,2.0,2,2,2,2,3.450305,3.332205,Bin2
4,Mistral,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,It seems like you've calculated the cost as if...,0.000623,0.004658,0.633301,0.302074,0.499444,72.665000,...,28,72.6650,13.0,2,2,2,2,4.707366,3.367296,Bin2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2471,Mistral,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,It seems there might be a misunderstanding in ...,0.000394,0.010368,0.562988,0.963009,0.461775,32.434286,...,22,32.4343,14.0,2,2,2,1,3.135494,3.135494,Bin1
2472,Phi3,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,"To solve this problem, we need to add the numb...",0.000448,0.009674,0.449219,0.657242,0.510343,69.788000,...,21,69.7880,10.0,0,0,0,0,4.138999,3.091042,Bin2
2473,Sonnet,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,That's a great start and I like how you worked...,0.000611,0.007957,0.683105,0.991297,0.586221,60.765000,...,33,60.7650,9.0,2,2,2,2,4.346788,3.526361,Bin4
2474,Expert,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,Okay. So Hector gave 5 less than four times as...,0.000407,0.006321,0.685059,0.129152,0.482296,71.767857,...,26,71.7679,7.0,2,2,2,2,3.659863,3.295837,Bin1


# STEP 2: LLM (Claude) Annotation

In [10]:
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("tama_replicate_key")

if secret_value_0:
    print('key gotten')
else:
    print('key not gotten')

key gotten


In [11]:
LOG_FILE = "classification_errors.log"
class TutorResponseAnnotator:
    def __init__(self, replicate_api_key: str, model: str= 'claude-3-sonnet-20240229'):
        self.replicate_api_token=replicate_api_key
        self.model=model
        self.client=replicate.Client(api_token=replicate_api_key)
        print(f"Replicate client initialized: {self.client is not None}")
    def create_prompt(self,  tutor_response: str, rubric:str, rubric_def:str, student_mistake:str)-> str:
        options=[0,1]
        prompt= f"""
        CONTEXT: 
        You are an expert annotator for math tutoring dialogue quality.        
        TASK:
        Rate the response on how much **{rubric}** is present in the tutor's response to a student's mistake {student_mistake} , your judgement should be solely based on the definition  below:
        Definition of {rubric}: {rubric_def}
        
        Tutor's response to evaluate:
        {tutor_response}
        
        Ratings Scale:
        1= Response satisfies the definition of {rubric}
        0= Response does not satisfy the definition of {rubric} 


        RESPONSE and  INSTRUCTIONS:
        1. Respond with only one integer rating  0 or 1
        2. No explanations, No additional text """
        


        return prompt.strip()
    def evaluate_response(self,conv_id: str, rubric:str,rubric_def:str, tutor_response: str,student_mistake:str,  max_tries:int =3) -> int:
 

        prompt = self.create_prompt(tutor_response,rubric,rubric_def,student_mistake)
        model_id="anthropic/claude-4.5-sonnet"
        for attempt in range(max_tries):
            try:
                print(f"  Analyzing (attempt {attempt + 1}/{max_tries})...")

                output = self.client.run(model_id,
                            input={
                                  "prompt":prompt,
                                  "max_tokens":1024,
                                  "temperature":0.0,
                                  "system": "You are an expert annotator for math tutoring dialogue quality.  Respond with only 0 or 1."
   
                              })
                response_text = ""
                for item in output:
                    response_text += item
                
                response = response_text.strip()
                
                if response in ['0', '1']:
                    return int(response)
                elif '0' in response:
                    return 0
                elif '1' in response:
                    return 1
                else:
                    print(f"     Unexpected response: '{response}'. Defaulting to 0")
                    return 0
            except Exception as e:
                print(f"     API error: {e}. Retrying...")
                time.sleep(2)
                
        with open(LOG_FILE, "a") as log:
            log.write(f"Retry limit exceeded | conv_id: {conv_id}, rubric: {rubric}, response: '{tutor_response[:100]}...'\n")

        return 0


In [12]:
def process_responses(path,rubrics,api_key):
    definitions={
    'politeness': " Language that conveys respect and consideration towards others. Politeness strategies may include courteous wording, acknowledgment, indirectness, softening, or cooperative interpersonal framing. ",  
    'agency': "Language that conveys initiative, purposeful action, or the capacity to act. Agentic language strategies may include expressions of planning, problem-solving, capability, persistence, or taking responsibility.",
    'PressAccuracy':"Language that seeks to verify, clarify, or improve the accuracy of a statement or claim. Pressing for accuracy strategies may include requests for clarification, verification, evidence, precision or correction.  ",
    'Uptake':" Language that reflects or builds on another person's contribution to support shared understanding and discussion. Uptake mechanisms may include paraphrasing, restatement, acknowledgment, validation, or extending another person's idea.",
    "PressReasoning":"Language that seeks to understand, explain, or justify the reasoning behind a statement or claim. Pressing for reasoning strategies may include requests for explanation, justification or elaboration.  "}
    annotator=TutorResponseAnnotator(api_key)
    results={}
    excel_file=pd.ExcelFile(path)
    for rubric in rubrics:
        print(f"Processing rubric: {rubric}")
        if rubric not in excel_file.sheet_names:
            print(f"Warning: Sheet '{rubric}' not found in {path}")
            print(f"Available sheets: {excel_file.sheet_names}")

            continue
        sample_df=pd.read_excel(path, sheet_name=rubric)
        #sample_df = sample_df.head(2).copy()

        print(f"Loaded {len(sample_df)} responses from sheet '{rubric}'")
            
        #test_sample=sample_df.head(5).copy()
        annotation_column = rubric
        rubric_def=definitions[rubric]
        sample_df[annotation_column] = None
        for idx, row in sample_df.iterrows():
            conv_id=str(row.get('conversation_id', f'row{idx}'))
            student_mistake=row['student_mistake']
            tutor_response=row['tutors_response']
            annotation=annotator.evaluate_response(
                    conv_id=conv_id,
                    rubric=rubric, 
                    rubric_def=rubric_def,
                    tutor_response=tutor_response,  
                    student_mistake=student_mistake,
                    max_tries=3)
            
            
            sample_df.at[idx, annotation_column] = annotation
                #print(f"  {rubric} annotation: {annotation}")
            time.sleep(0.5)
        results[rubric] = sample_df
        output_file = f"annotated_{rubric}.csv"
        sample_df.to_csv(output_file, index=False)
    return results

In [ ]:
if __name__ == "__main__":
    user_secrets = UserSecretsClient()
    replicate_api_key = user_secrets.get_secret("tama_replicate_key")
    rubrics = ['politeness', 'agency', 'PressReasoning', 'PressAccuracy', 'Uptake']
    path = "/kaggle/input/datasets/abdulsalamramatu/100-feature-response-sample/total_response_sample.xlsx"
    annotated_results = process_responses(path, rubrics, replicate_api_key)

# STEP 3: HUMAN / LLM Agreement


In [4]:
path="/kaggle/input/datasets/abdulsalamramatu/human-llm-annotations/ARR_Revise_Resubmit_HUMAN_LLM_Annotations.xlsx"
rubrics = ['Politeness', 'Agency', 'PressReasoning', 'PressAccuracy', 'Uptake']
validation_results={}
for rubric in rubrics:
    print(f"Processing rubric: {rubric}")
    man_llm_df=pd.read_excel(path, sheet_name=rubric)
    
    annot_match=man_llm_df[man_llm_df['human']==man_llm_df['LLM']].copy()
    print(f"Agreements: {len(annot_match)} ({len(annot_match)/len(man_llm_df)*100:.1f}%)")

    annot_mismatch = man_llm_df[man_llm_df['human'] != man_llm_df['LLM']].copy()
    print(f"Disagreements: {len(annot_mismatch)} ({len(annot_mismatch)/len(man_llm_df)*100:.1f}%)")

    if len(annot_mismatch) > 0:
        disagreement_file = f"disagreements_{rubric}.csv"
        annot_mismatch.to_csv(disagreement_file, index=False)
        print(f"Saved {len(annot_mismatch)} disagreements to {disagreement_file}")

Processing rubric: Politeness
Agreements: 88 (88.0%)
Disagreements: 12 (12.0%)
Saved 12 disagreements to disagreements_Politeness.csv
Processing rubric: Agency
Agreements: 38 (38.0%)
Disagreements: 62 (62.0%)
Saved 62 disagreements to disagreements_Agency.csv
Processing rubric: PressReasoning
Agreements: 28 (28.0%)
Disagreements: 72 (72.0%)
Saved 72 disagreements to disagreements_PressReasoning.csv
Processing rubric: PressAccuracy
Agreements: 73 (73.0%)
Disagreements: 27 (27.0%)
Saved 27 disagreements to disagreements_PressAccuracy.csv
Processing rubric: Uptake
Agreements: 71 (71.0%)
Disagreements: 29 (29.0%)
Saved 29 disagreements to disagreements_Uptake.csv


# STEP 4: EVALUATE PERFORMANCE

In [ ]:
features_df['conversation_id'] = features_df['conversation_id'].astype(str)
features_df['tutor'] = features_df['tutor'].astype(str).str.strip()

In [53]:
human_llm_annot_path="/kaggle/input/datasets/abdulsalamramatu/human-llm-annotations/ARR_Revise_Resubmit_HUMAN_LLM_Annotations.xlsx"
resolve_path="/kaggle/input/datasets/abdulsalamramatu/resolved/1disagreement (1).xlsx"
#math_df_path="/kaggle/input/datasets/abdulsalamramatu/complete-math-features/new_math_df.csv"

eval_results={}

In [56]:
#merging annot_prob_resolve
all_results = {}

for rubric in rubrics:
    print(f"Processing rubric: {rubric}")
    annotations_df = pd.read_excel(human_llm_annot_path, sheet_name=rubric)
    
    resolved_df = pd.read_excel(resolve_path, sheet_name=rubric)
    
    annotations_df['conversation_id'] = annotations_df['conversation_id'].astype(str)
    annotations_df['tutor'] = annotations_df['tutor'].astype(str).str.strip()
    
    if 'conversation_id' in resolved_df.columns:
        resolved_df['conversation_id'] = resolved_df['conversation_id'].astype(str)
    
    feature_col = rubric_to_feature_col[rubric]
    
    eval_df = annotations_df.merge(
        features_df[['conversation_id', 'tutor', feature_col]],
        on=['conversation_id', 'tutor'],
        how='left'
    )
    
    eval_df = eval_df.rename(columns={feature_col: 'classifier_probability'})
    
    print(f"Total annotations: {len(eval_df)}")
    print(f"Found probabilities: {eval_df['classifier_probability'].notna().sum()}")
    print(f"Missing probabilities: {eval_df['classifier_probability'].isna().sum()}")
    
    if 'conversation_id' in resolved_df.columns and 'resolve' in resolved_df.columns:
        eval_df = eval_df.merge(
            resolved_df[['conversation_id', 'resolve']],
            on='conversation_id',
            how='left'
        )
        print(f"Merged resolved disagreements")
    else:
        eval_df['resolve'] = np.nan
    
 
    eval_df['label'] = np.where(
        eval_df['human'] == eval_df['LLM'],
        eval_df['human'],
        eval_df['resolve']
    )
    
 
    

    column_order = [
        'conversation_id', 
        'tutor', 
        'student_mistake', 
        'tutors_response',
        'human', 
        'LLM', 
        'resolve',
        'label',
        'classifier_probability'
    ]
    
    existing_columns = [col for col in column_order if col in eval_df.columns]
    final_df = eval_df[existing_columns]
    

 
    
   
    all_results[rubric] = final_df
  


print("All rubrics processed successfully!")

with pd.ExcelWriter("y.xlsx") as writer:
    for rubric, df in all_results.items():
        df.to_excel(writer, sheet_name=rubric, index=False)



Processing rubric: Politeness
Total annotations: 100
Found probabilities: 100
Missing probabilities: 0
Merged resolved disagreements

Processing rubric: Agency
Total annotations: 100
Found probabilities: 100
Missing probabilities: 0
Merged resolved disagreements

Processing rubric: PressReasoning
Total annotations: 100
Found probabilities: 100
Missing probabilities: 0
Merged resolved disagreements

Processing rubric: PressAccuracy
Total annotations: 100
Found probabilities: 100
Missing probabilities: 0
Merged resolved disagreements

Processing rubric: Uptake
Total annotations: 100
Found probabilities: 100
Missing probabilities: 0
Merged resolved disagreements

All rubrics processed successfully!


In [8]:
features = ['PressReasoning_prob', 'PressAccuracy_prob', 'Uptake_prob', 'politeness_score', 'agency_score', 'PressReasoning',]
rubrics = ['Politeness', 'Agency',  'PressAccuracy', 'Uptake','PressReasoning']

rubric_to_feature_col={'Politeness': 'politeness_score',
                       'Agency': 'agency_score',        
                        'PressAccuracy': 'PressAccuracy_prob',
                       'PressReasoning': 'PressReasoning_prob',
                        'Uptake': 'Uptake_prob'}
for rubric, feature_col in rubric_to_feature_col.items():
    print(f"  {rubric} → {feature_col}") 


  Politeness → politeness_score
  Agency → agency_score
  PressAccuracy → PressAccuracy_prob
  PressReasoning → PressReasoning_prob
  Uptake → Uptake_prob


In [9]:
combined_file = "/kaggle/input/datasets/abdulsalamramatu/full-resolve/1all_annotations_with_probabilities (2).xlsx"
auc_summary = {}
kappa_summary = {}
agreement_summary = {}
for rubric in rubrics:
    
    print(f"EVALUATING RUBRIC: {rubric.upper()}")
    eval_df = pd.read_excel(combined_file, sheet_name=rubric)
    eval_df = eval_df.dropna(subset=['label', 'classifier_probability']).copy()
    

    eval_df['label'] = eval_df['label'].astype(int)
    unique_classes = eval_df['label'].unique()

    agreement_df = eval_df.dropna(subset=['human', 'LLM'])

    if len(agreement_df) > 0:
        print(f"agreement_df={len(agreement_df)}")
        agreement_rate = (agreement_df['human'] == agreement_df['LLM']).mean()
        kappa = cohen_kappa_score(agreement_df['human'], agreement_df['LLM'])
        
        print(f"Agreement Rate: {agreement_rate:.3f} ({agreement_rate*100:.1f}%)")
        print(f"Cohen's Kappa: {kappa:.3f}")
        
        if kappa < 0:
            interpretation = "Poor (worse than chance)"
        elif kappa < 0.20:
            interpretation = "Slight"
        elif kappa < 0.40:
            interpretation = "Fair"
        elif kappa < 0.60:
            interpretation = "Moderate"
        elif kappa < 0.80:
            interpretation = "Substantial"
        else:
            interpretation = "Almost perfect"
        print(f"Interpretation: {interpretation} agreement")
        
        agreement_summary[rubric] = agreement_rate
        kappa_summary[rubric] = kappa
    
    unique_classes = eval_df['label'].unique()
    print(f"Classes present: {unique_classes}")
    
    if len(unique_classes) == 2:
        auc = roc_auc_score(eval_df['label'], eval_df['classifier_probability'])
        print(f"AUC: {auc:.3f}")
        auc_summary[rubric] = auc
    else:
        print(f"Cannot calculate AUC - need both classes (0 and 1)")
        print(f"   Only class {unique_classes[0]} present")
    
    # Classifier accuracy at 0.5 threshold
    eval_df['classifier_prediction'] = (eval_df['classifier_probability'] >= 0.5).astype(int)
    classifier_acc = accuracy_score(eval_df['label'], eval_df['classifier_prediction'])
    print(f"Classifier Accuracy (threshold=0.5): {classifier_acc:.3f}")
    print()

summary_df = pd.DataFrame({
    'Rubric': list(agreement_summary.keys()),
    'Agreement_Rate': [agreement_summary.get(r, np.nan) for r in agreement_summary.keys()],
    "Cohen's_Kappa": [kappa_summary.get(r, np.nan) for r in kappa_summary.keys()],
    'AUC': [auc_summary.get(r, np.nan) for r in auc_summary.keys()]
})

print("\n" + summary_df.to_string(index=False))
summary_df.to_csv("evaluation_summary.csv", index=False)
print("\n Saved evaluation_summary.csv")


EVALUATING RUBRIC: POLITENESS
agreement_df=100
Agreement Rate: 0.880 (88.0%)
Cohen's Kappa: 0.437
Interpretation: Moderate agreement
Classes present: [0 1]
AUC: 0.887
Classifier Accuracy (threshold=0.5): 0.830

EVALUATING RUBRIC: AGENCY
agreement_df=100
Agreement Rate: 0.380 (38.0%)
Cohen's Kappa: 0.086
Interpretation: Slight agreement
Classes present: [1 0]
AUC: 0.493
Classifier Accuracy (threshold=0.5): 0.570

EVALUATING RUBRIC: PRESSACCURACY
agreement_df=100
Agreement Rate: 0.730 (73.0%)
Cohen's Kappa: 0.311
Interpretation: Fair agreement
Classes present: [1 0]
AUC: 0.464
Classifier Accuracy (threshold=0.5): 0.290

EVALUATING RUBRIC: UPTAKE
agreement_df=100
Agreement Rate: 0.710 (71.0%)
Cohen's Kappa: 0.338
Interpretation: Fair agreement
Classes present: [0 1]
AUC: 0.842
Classifier Accuracy (threshold=0.5): 0.650

EVALUATING RUBRIC: PRESSREASONING
agreement_df=100
Agreement Rate: 0.280 (28.0%)
Cohen's Kappa: 0.011
Interpretation: Slight agreement
Classes present: [0 1]
AUC: 0.796
Cl

# STEP 4a: HUMAN AND LLM ANNOTATIONS SEPERATELY

In [63]:
human_auc_summary = {}
llm_auc_summary = {}
agreement_summary = {}
kappa_summary = {}
for rubric in rubrics:
    print(f"EVALUATING RUBRIC: {rubric.upper()}")
    df = pd.read_excel(combined_file, sheet_name=rubric)
    eval_df = df.dropna(subset=['classifier_probability']).copy()
    agreement_df = eval_df.dropna(subset=['human', 'LLM'])
    
    if len(agreement_df) > 0:
        agreement_rate = (agreement_df['human'] == agreement_df['LLM']).mean()
        kappa = cohen_kappa_score(agreement_df['human'], agreement_df['LLM'])
        
        print(f"Inter-Annotator Agreement:")
        print(f"Agreement Rate: {agreement_rate:.3f} ({agreement_rate*100:.1f}%)")
        print(f"Cohen's Kappa: {kappa:.3f}")
        
        agreement_summary[rubric] = agreement_rate
        kappa_summary[rubric] = kappa
        #human as groundtruth
        human_df = eval_df.dropna(subset=['human']).copy()
        print(f"Samples with human annotation: {len(human_df)}")
        
        human_classes = human_df['human'].unique()
        auc_human = roc_auc_score(human_df['human'], human_df['classifier_probability'])
        print(f"AUC (Human as Ground Truth): {auc_human:.3f}")
        human_auc_summary[rubric] = auc_human

        #llm as groundtruth
        llm_df = eval_df.dropna(subset=['LLM']).copy()
        llm_classes = llm_df['LLM'].unique()
        print(f"Samples with LLM annotation: {len(llm_df)}")
        auc_llm = roc_auc_score(llm_df['LLM'], llm_df['classifier_probability'])
        print(f"AUC (LLM as Ground Truth): {auc_llm:.3f}")
        llm_auc_summary[rubric] = auc_llm
        print()
        comparison_df = pd.DataFrame({
            'Rubric': list(human_auc_summary.keys()),
            'AUC (Human GT)': [human_auc_summary.get(r, np.nan) for r in human_auc_summary.keys()],
            'AUC (LLM GT)': [llm_auc_summary.get(r, np.nan) for r in llm_auc_summary.keys()],
            'Agreement_Rate': [agreement_summary.get(r, np.nan) for r in agreement_summary.keys()],
            "Cohen's_Kappa": [kappa_summary.get(r, np.nan) for r in kappa_summary.keys()]
})
comparison_df.to_csv("auc_comparison_human_vs_llm.csv", index=False)


EVALUATING RUBRIC: POLITENESS
Inter-Annotator Agreement:
Agreement Rate: 0.880 (88.0%)
Cohen's Kappa: 0.437
Samples with human annotation: 100
AUC (Human as Ground Truth): 0.756
Samples with LLM annotation: 100
AUC (LLM as Ground Truth): 0.867

EVALUATING RUBRIC: AGENCY
Inter-Annotator Agreement:
Agreement Rate: 0.380 (38.0%)
Cohen's Kappa: 0.086
Samples with human annotation: 100
AUC (Human as Ground Truth): 0.375
Samples with LLM annotation: 100
AUC (LLM as Ground Truth): 0.652

EVALUATING RUBRIC: PRESSREASONING
Inter-Annotator Agreement:
Agreement Rate: 0.280 (28.0%)
Cohen's Kappa: 0.011
Samples with human annotation: 100
AUC (Human as Ground Truth): 0.762
Samples with LLM annotation: 100
AUC (LLM as Ground Truth): 0.764

EVALUATING RUBRIC: PRESSACCURACY
Inter-Annotator Agreement:
Agreement Rate: 0.730 (73.0%)
Cohen's Kappa: 0.311
Samples with human annotation: 100
AUC (Human as Ground Truth): 0.517
Samples with LLM annotation: 100
AUC (LLM as Ground Truth): 0.359

EVALUATING RUBRIC

# STEP 5: SAMPLING TOP AND BOTTOM RESPONSES


In [13]:
extreme_only_results = {}
for rubric in rubrics:
    df = pd.read_excel(combined_file, sheet_name=rubric)
    score_col = "classifier_probability"
    bin_edges = np.percentile(df[score_col], [0, 25, 50, 75, 100])
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf
    df['score_bin'] = pd.cut(
        df[score_col],
        bins=bin_edges,
        labels=['Q1', 'Q2', 'Q3', 'Q4'],
        include_lowest=True
    )

    extreme_df = df[df['score_bin'].isin(['Q1 (Lowest)', 'Q4 (Highest)'])].copy()
    extreme_only_results[rubric] = extreme_df

    output_file = f"extreme_only_{rubric.lower()}.csv"
    extreme_df.to_csv(output_file, index=False)

    with pd.ExcelWriter("extreme_only_all_rubrics.xlsx") as writer:
        for rubric, df in extreme_only_results.items():
            if 'score_bin' in df.columns:
                df_to_save = df.drop(columns=['score_bin'])
            else:
                df_to_save = df
            df_to_save.to_excel(writer, sheet_name=rubric, index=False)

print(" Saved to extreme_only_all_rubrics.xlsx")

 Saved to extreme_only_all_rubrics.xlsx


# Step 6: AGREEMENT HUMAN AND LLM

In [17]:
combined_file = "/kaggle/input/datasets/abdulsalamramatu/extreme-samples/extreme_only_all_rubrics (1).xlsx"
auc_summary = {}
kappa_summary = {}
agreement_summary = {}
for rubric in rubrics:
    
    print(f"EVALUATING RUBRIC: {rubric.upper()}")
    eval_df = pd.read_excel(combined_file, sheet_name=rubric)
    eval_df = eval_df.dropna(subset=['label', 'classifier_probability']).copy()
    

    eval_df['label'] = eval_df['label'].astype(int)
    unique_classes = eval_df['label'].unique()

    agreement_df = eval_df.dropna(subset=['human', 'LLM'])

    if len(agreement_df) > 0:
        print(f"agreement_df={len(agreement_df)}")
        agreement_rate = (agreement_df['human'] == agreement_df['LLM']).mean()
        kappa = cohen_kappa_score(agreement_df['human'], agreement_df['LLM'])
        
        print(f"Agreement Rate: {agreement_rate:.3f} ({agreement_rate*100:.1f}%)")
        print(f"Cohen's Kappa: {kappa:.3f}")
        
        if kappa < 0:
            interpretation = "Poor (worse than chance)"
        elif kappa < 0.20:
            interpretation = "Slight"
        elif kappa < 0.40:
            interpretation = "Fair"
        elif kappa < 0.60:
            interpretation = "Moderate"
        elif kappa < 0.80:
            interpretation = "Substantial"
        else:
            interpretation = "Almost perfect"
        print(f"Interpretation: {interpretation} agreement")
        
        agreement_summary[rubric] = agreement_rate
        kappa_summary[rubric] = kappa
    
    unique_classes = eval_df['label'].unique()
    print(f"Classes present: {unique_classes}")
    
    if len(unique_classes) == 2:
        auc = roc_auc_score(eval_df['label'], eval_df['classifier_probability'])
        print(f"AUC: {auc:.3f}")
        auc_summary[rubric] = auc
    else:
        print(f"Cannot calculate AUC - need both classes (0 and 1)")
        print(f"   Only class {unique_classes[0]} present")
    
    # Classifier accuracy at 0.5 threshold
    eval_df['classifier_prediction'] = (eval_df['classifier_probability'] >= 0.5).astype(int)
    classifier_acc = accuracy_score(eval_df['label'], eval_df['classifier_prediction'])
    print(f"Classifier Accuracy (threshold=0.5): {classifier_acc:.3f}")
    print()

summary_df = pd.DataFrame({
    'Rubric': list(agreement_summary.keys()),
    'Agreement_Rate': [agreement_summary.get(r, np.nan) for r in agreement_summary.keys()],
    "Cohen's_Kappa": [kappa_summary.get(r, np.nan) for r in kappa_summary.keys()],
    'AUC': [auc_summary.get(r, np.nan) for r in auc_summary.keys()]
})

print("\n" + summary_df.to_string(index=False))
summary_df.to_csv("extreme_human_n_llm_evaluation_summary.csv", index=False)
print("\n Saved extreme human and llm evaluation_summary.csv")


EVALUATING RUBRIC: POLITENESS
agreement_df=50
Agreement Rate: 0.880 (88.0%)
Cohen's Kappa: 0.598
Interpretation: Moderate agreement
Classes present: [0 1]
AUC: 0.808
Classifier Accuracy (threshold=0.5): 0.700

EVALUATING RUBRIC: AGENCY
agreement_df=50
Agreement Rate: 0.440 (44.0%)
Cohen's Kappa: 0.138
Interpretation: Slight agreement
Classes present: [1 0]
AUC: 0.402
Classifier Accuracy (threshold=0.5): 0.440

EVALUATING RUBRIC: PRESSACCURACY
agreement_df=50
Agreement Rate: 0.740 (74.0%)
Cohen's Kappa: 0.129
Interpretation: Slight agreement
Classes present: [1 0]
AUC: 0.560
Classifier Accuracy (threshold=0.5): 0.340

EVALUATING RUBRIC: UPTAKE
agreement_df=50
Agreement Rate: 0.640 (64.0%)
Cohen's Kappa: 0.265
Interpretation: Fair agreement
Classes present: [0 1]
AUC: 0.872
Classifier Accuracy (threshold=0.5): 0.700

EVALUATING RUBRIC: PRESSREASONING
agreement_df=50
Agreement Rate: 0.420 (42.0%)
Cohen's Kappa: 0.057
Interpretation: Slight agreement
Classes present: [0 1]
AUC: 0.869
Class

In [18]:

human_auc_summary = {}
llm_auc_summary = {}
agreement_summary = {}
kappa_summary = {}
for rubric in rubrics:
    print(f"EVALUATING RUBRIC: {rubric.upper()}")
    df = pd.read_excel(combined_file, sheet_name=rubric)
    eval_df = df.dropna(subset=['classifier_probability']).copy()
    agreement_df = eval_df.dropna(subset=['human', 'LLM'])
    
    if len(agreement_df) > 0:
        agreement_rate = (agreement_df['human'] == agreement_df['LLM']).mean()
        kappa = cohen_kappa_score(agreement_df['human'], agreement_df['LLM'])
        
        print(f"Inter-Annotator Agreement:")
        print(f"Agreement Rate: {agreement_rate:.3f} ({agreement_rate*100:.1f}%)")
        print(f"Cohen's Kappa: {kappa:.3f}")
        
        agreement_summary[rubric] = agreement_rate
        kappa_summary[rubric] = kappa
        #human as groundtruth
        human_df = eval_df.dropna(subset=['human']).copy()
        print(f"Samples with human annotation: {len(human_df)}")
        
        human_classes = human_df['human'].unique()
        auc_human = roc_auc_score(human_df['human'], human_df['classifier_probability'])
        print(f"AUC (Human as Ground Truth): {auc_human:.3f}")
        human_auc_summary[rubric] = auc_human

        #llm as groundtruth
        llm_df = eval_df.dropna(subset=['LLM']).copy()
        llm_classes = llm_df['LLM'].unique()
        print(f"Samples with LLM annotation: {len(llm_df)}")
        auc_llm = roc_auc_score(llm_df['LLM'], llm_df['classifier_probability'])
        print(f"AUC (LLM as Ground Truth): {auc_llm:.3f}")
        llm_auc_summary[rubric] = auc_llm
        print()
        comparison_df = pd.DataFrame({
            'Rubric': list(human_auc_summary.keys()),
            'AUC (Human GT)': [human_auc_summary.get(r, np.nan) for r in human_auc_summary.keys()],
            'AUC (LLM GT)': [llm_auc_summary.get(r, np.nan) for r in llm_auc_summary.keys()],
            'Agreement_Rate': [agreement_summary.get(r, np.nan) for r in agreement_summary.keys()],
            "Cohen's_Kappa": [kappa_summary.get(r, np.nan) for r in kappa_summary.keys()]
})
comparison_df.to_csv("auc_comparison_extreme_human_vs_llm.csv", index=False)



EVALUATING RUBRIC: POLITENESS
Inter-Annotator Agreement:
Agreement Rate: 0.880 (88.0%)
Cohen's Kappa: 0.598
Samples with human annotation: 50
AUC (Human as Ground Truth): 0.738
Samples with LLM annotation: 50
AUC (LLM as Ground Truth): 0.821

EVALUATING RUBRIC: AGENCY
Inter-Annotator Agreement:
Agreement Rate: 0.440 (44.0%)
Cohen's Kappa: 0.138
Samples with human annotation: 50
AUC (Human as Ground Truth): 0.343
Samples with LLM annotation: 50
AUC (LLM as Ground Truth): 0.542

EVALUATING RUBRIC: PRESSACCURACY
Inter-Annotator Agreement:
Agreement Rate: 0.740 (74.0%)
Cohen's Kappa: 0.129
Samples with human annotation: 50
AUC (Human as Ground Truth): 0.614
Samples with LLM annotation: 50
AUC (LLM as Ground Truth): 0.326

EVALUATING RUBRIC: UPTAKE
Inter-Annotator Agreement:
Agreement Rate: 0.640 (64.0%)
Cohen's Kappa: 0.265
Samples with human annotation: 50
AUC (Human as Ground Truth): 0.827
Samples with LLM annotation: 50
AUC (LLM as Ground Truth): 0.603

EVALUATING RUBRIC: PRESSREASONING